# 15 — The discriminating experiment

This is the experiment the capstone has been preparing. In `04/14` you opened a second channel in the joint state $\rho_{AB}$ — the second circular moment of the phase difference, a quantity the phase-locking value cannot see. Today you put that channel to a controlled test: you build two systems that PLV is unable to tell apart, you watch a richer quantum coupling measure separate them, and then you do what a careful experimenter does next — you set a classical statistic of the same second moment beside it and score every measure with one protocol.

**Prerequisites:** `14_richer_embedding_applied` (the multi-frequency qutrit and its second-moment channel) and `13_the_redundancy_reveal` (on the naive embedding the quantum measure can only echo PLV).

**What you'll be able to do**
- Construct two ensembles with **identical PLV by construction** — same first circular moment $a_1$ — but a **different second moment** $a_2$, using `matched_plv_ensembles`.
- Confirm that PLV and the **naive** phase-qubit embedding's quantum mutual information are both blind to the difference.
- Show that the **richer multi-frequency embedding's** quantum mutual information **separates** the two ensembles.
- Compute the **second-harmonic phase-locking value**, the classical statistic of the same second moment, and see that it separates them too.
- Score all four statistics with **one protocol** — a signed score against a null of 100 draws, an area under the curve and a detection rate, repeated over five seeds — and read off **how many samples each statistic needs**.

You will leave with a result you can state to the digit and defend.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.bench_protocol import (
    auc,
    detection_rate,
    harmonic_plv,
    signed_score,
)
from qot_course.quantum_ot.capstone import (
    plv,
    joint_density_matrix,
    coupling_qmi,
)
from qot_course.quantum_ot.embeddings import (
    multifreq_state,
    joint_density_from_states,
)
from qot_course.quantum_ot.discrimination import matched_plv_ensembles

np.random.seed(42)
viz.use_course_style()


def rich_qmi(theta_a, theta_b):
    """Quantum mutual information of the multi-frequency (qutrit) joint embedding.

    Embeds each phase record with multifreq_state(harmonics=(1, 2)) -> a 3-level state
    carrying the first AND second circular moments, joins them into the 9x9 rho_AB, and
    returns I(A:B) in nats. This is the richer-embedding quantum coupling measure that the
    experiment sets beside the phase-locking value.
    """
    rho = joint_density_from_states(multifreq_state(theta_a), multifreq_state(theta_b))
    return coupling_qmi(rho, dims=(3, 3))


# Sample count per channel. Large enough that PLV and the circular moments are tight
# Monte-Carlo estimates (error ~ 1/sqrt(n)), small enough that the notebook stays fast.
N = 150_000

## 1. The setup: two systems PLV cannot tell apart

Notebook `04/14` left us with a tool and a promise. The tool: the multi-frequency qutrit, whose joint density $\rho_{AB}$ carries the phase difference's **second** circular moment $\langle e^{2i\delta}\rangle$ in an element — $\rho_{AB}[6,2]$ — that the naive phase qubit does not even possess. The promise: that this second channel is an *independent* degree of freedom, so two couplings can share a PLV yet differ in it. Today we collect on that promise.

We build two ensembles, *low* and *high*, that are matched where PLV looks and parted where it cannot. Both draw their phase difference $\delta = \theta_A - \theta_B$ from the truncated-Fourier circular density

$$ p(\delta) \;\propto\; 1 + 2\,a_1 \cos\delta + 2\,a_2 \cos 2\delta, \qquad \delta \in [0, 2\pi), $$

whose first two circular moments are *exactly* $\langle e^{i\delta}\rangle = a_1$ and $\langle e^{2i\delta}\rangle = a_2$ (Mardia & Jupp 2000: the order-$k$ Fourier coefficient of a wrapped density is its $k$-th circular moment). We give both ensembles the **same $a_1 = 0.4$** — so they share their first moment, hence their PLV — but **different second moments**: $a_2 = 0.0$ for *low* (no second-order structure) and $a_2 = 0.3$ for *high*. A single uniform reference phase $\theta_A$, shared by both, keeps each single-channel marginal near-uniform, so the two ensembles differ in **nothing but** their joint second-order structure.

This is the cleanest possible test of the thesis. If a measure separates these two ensembles, it must be reading something past the first moment — because that is the only thing we let differ.

In [ ]:
# Two ensembles: identical first moment a1 (-> identical PLV), different second moment a2.
# matched_plv_ensembles shares one uniform reference phase theta_A across both dyads and
# draws independent phase differences (seed -> theta_A, seed+1 -> low delta, seed+2 -> high).
(theta_a_low, theta_b_low), (theta_a_high, theta_b_high) = matched_plv_ensembles(
    N, a1=0.4, a2_low=0.0, a2_high=0.3, seed=0
)

# The reference phase theta_A is shared, so the two ensembles differ only through delta.
print(f"samples per channel       N = {N:,}")
print(f"shared reference phase    theta_A identical across ensembles? "
      f"{np.array_equal(theta_a_low, theta_a_high)}")
print()
print("                          a1 (first moment)   a2 (second moment)")
print(f"low  ensemble             {0.4:>10.2f}        {0.0:>14.2f}")
print(f"high ensemble             {0.4:>10.2f}        {0.3:>14.2f}")

**Read the output.** Two ensembles, each $N$ phase samples per channel, sharing one reference phase $\theta_A$ (the `True` confirms it), so that *only* the phase difference $\delta$ — and within it, *only* the second moment $a_2$ — distinguishes them. Both carry the same first moment $a_1 = 0.4$; the low ensemble has $a_2 = 0$ and the high one $a_2 = 0.3$. Everything that follows asks each measure in turn whether it can hear this single, deliberate difference.

## 2. PLV is matched — by construction

The first measure is the neuroscience standard, the phase-locking value $\mathrm{PLV} = |\langle e^{i(\theta_A - \theta_B)}\rangle| = |\langle e^{i\delta}\rangle|$ (Lachaux et al. 1999). PLV is, definitionally, the magnitude of the **first** circular moment of the phase difference. We built both ensembles with the same $a_1$, so PLV must report the same value for both — up to the Monte-Carlo wobble of a finite sample.

We compute it for each ensemble and read off the gap.

In [ ]:
plv_low = plv(theta_a_low, theta_b_low)
plv_high = plv(theta_a_high, theta_b_high)

print(f"PLV (low ensemble,  a2 = 0.0)   {plv_low:.4f}")
print(f"PLV (high ensemble, a2 = 0.3)   {plv_high:.4f}")
print(f"PLV gap                         {abs(plv_low - plv_high):.4f}")

**Read the output.** The two PLV values agree to a few thousandths, and the residual is the $1/\sqrt{N}$ Monte-Carlo wobble of two finite samples that share $a_1$ exactly. **By construction, PLV cannot distinguish these two systems.** To the phase-locking value they are the same dyad. If the second-order difference we built in is to be detected at all, it must be by a measure that looks past the first moment — and the rest of the notebook asks which measures can.

## 3. The naive quantum embedding is blind too

Before trusting any quantum measure, we must rule out a natural worry: that *moving to a quantum state is already enough to see the difference*. It is not — and `04/13` told us why. The naive phase qubit $(|0\rangle + e^{i\theta}|1\rangle)/\sqrt 2$ carries only the first harmonic $e^{i\theta}$, so the single coupling-bearing coherence of its joint $\rho_{AB}$ is the first circular moment — the very quantity that *is* PLV. A quantum measure reading that state inherits PLV's blindness exactly.

So we compute the quantum mutual information of the **naive** $4\times 4$ joint density for each ensemble. If `04/13`'s diagnosis is right, the naive QMI must be matched here too.

In [ ]:
naive_qmi_low = coupling_qmi(joint_density_matrix(theta_a_low, theta_b_low))
naive_qmi_high = coupling_qmi(joint_density_matrix(theta_a_high, theta_b_high))

print("naive phase-qubit embedding  (4x4 joint density, QMI in nats)")
print(f"  QMI (low ensemble,  a2 = 0.0)   {naive_qmi_low:.5f}")
print(f"  QMI (high ensemble, a2 = 0.3)   {naive_qmi_high:.5f}")
print(f"  QMI gap                         {abs(naive_qmi_low - naive_qmi_high):.5f}")

**Read the output.** The naive embedding's quantum mutual information is matched between the two ensembles — its gap sits in the ten-thousandths of a nat, the Monte-Carlo floor of this measure. This is the redundancy `04/13` diagnosed: because the naive qubit's only coupling channel *is* the first circular moment, the naive QMI is a re-encoding of PLV and inherits its blindness to $a_2$. Switching from a classical phase statistic to a quantum state, on its own, buys nothing here. Whatever separation comes next must come from the **richer** embedding, not from the word *quantum*.

## 4. The richer embedding separates them

Now the measure built for the job. We embed each phase record with the multi-frequency qutrit $\tfrac{1}{\sqrt 3}\bigl(|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle\bigr)$ from `04/14`, join the two qutrits into the $9\times 9$ density $\rho_{AB}$, and take its quantum mutual information (this is the `rich_qmi` helper from the imports). The qutrit carries the second harmonic $e^{2i\theta}$, so its $\rho_{AB}$ holds the second circular moment $\langle e^{2i\delta}\rangle$ in the coherence $\rho_{AB}[6,2]$ — the channel the naive qubit lacked. The low and high ensembles differ precisely in that moment. If the richer-embedding QMI is doing what `04/14` promised, the two values should now come apart.

In [ ]:
rich_qmi_low = rich_qmi(theta_a_low, theta_b_low)
rich_qmi_high = rich_qmi(theta_a_high, theta_b_high)
rich_gap = abs(rich_qmi_low - rich_qmi_high)

print("richer multi-frequency embedding  (9x9 qutrit joint density, QMI in nats)")
print(f"  QMI (low ensemble,  a2 = 0.0)   {rich_qmi_low:.5f}")
print(f"  QMI (high ensemble, a2 = 0.3)   {rich_qmi_high:.5f}")
print(f"  QMI gap                         {rich_gap:.5f}")
print()
print("for contrast, the gaps the blind measures reported:")
print(f"  PLV gap                         {abs(plv_low - plv_high):.5f}")
print(f"  naive-embedding QMI gap         {abs(naive_qmi_low - naive_qmi_high):.5f}")

**Read the output.** The richer embedding's quantum mutual information is **different** for the two ensembles: 0.0787 nat on the high ensemble ($a_2 = 0.3$) against 0.0743 nat on the low one ($a_2 = 0.0$), a gap of 0.0044 nat. The naive embedding's gap, in the same unit, sat at its Monte-Carlo floor of 0.0004 nat. (The PLV gap is printed for reference only: PLV is a dimensionless modulus, so its gap does not live on the scale of a gap in nats.) The second channel of `04/14` is reading something that PLV and the naive quantum embedding do not.

Two things remain before this becomes a result. A single pair of different numbers is not yet evidence, because any two finite samples differ a little: sections 6 and 7 settle that for every statistic at once. And a fair experimenter asks whether a *classical* statistic reads the same second moment: that is section 5.

## 5. The classical neighbour: a second-harmonic phase-locking value

In `04/14` we set out to turn the second channel into a measured separation, and section 4 gave us a first one. Before we call it a result, we give it the comparison that decides how far it reaches: a separation that has met its closest classical rival is worth more than one that never did.

The two ensembles differ in one thing, the second circular moment $\langle e^{2i\delta}\rangle$. The phase-locking value is the modulus of the *first* moment. Nothing stops us from taking the modulus of the *second*:

$$ \mathrm{PLV}_2 \;=\; \bigl|\langle e^{2i(\theta_A - \theta_B)}\rangle\bigr| . $$

This is a classical statistic, and it costs what PLV costs: one line of arithmetic on the same phases, with no density matrix. It is the modulus of the sample second trigonometric moment of circular statistics (Mardia & Jupp 2000), and the natural classical neighbour of the richer embedding: the qutrit of `04/14` writes the first and the second harmonic into the state, and PLV and $\mathrm{PLV}_2$ read those same two harmonics directly from the phases.

In [ ]:
plv2_low = harmonic_plv(theta_a_low, theta_b_low, harmonic=2)
plv2_high = harmonic_plv(theta_a_high, theta_b_high, harmonic=2)

print("second-harmonic phase-locking value  |<exp(2i delta)>|")
print(f"  low ensemble  (a2 = 0.0)        {plv2_low:.5f}")
print(f"  high ensemble (a2 = 0.3)        {plv2_high:.5f}")
print(f"  gap                             {abs(plv2_low - plv2_high):.5f}")
print()
print("for contrast, the gap of the richer quantum measure (section 4):")
print(f"  rich-QMI gap                    {rich_gap:.5f}")

**Read the output.** The second-harmonic phase-locking value is 0.004 on the low ensemble and 0.302 on the high one: it returns the two values of $a_2$ we built in, 0.0 and 0.3, to within sampling error. The classical statistic sees the difference, and it sees it large — a gap of 0.298 on a scale that runs from 0 to 1. The richer quantum measure sees the same difference as a gap of 0.0044 nat, on top of a value near 0.074 nat.

These two gaps are in different units, so we cannot yet say which statistic separates the ensembles *better*: that takes a common yardstick, the work of the next section. What we can already say sharpens section 4: the difference the richer quantum measure detects is **not** hidden from classical statistics. It was hidden from the *first-harmonic* phase-locking value.

## 6. One protocol for every statistic

To compare statistics that live on different scales, we ask each of them the same three questions, on the same draws.

Each draw gives one *low* and one *high* ensemble, hence one **observed difference** (high minus low) per statistic. Beside it we draw two *low* ensembles — same $a_2$, nothing to find — whose difference is a **null difference**: what the statistic reports from sampling alone. These are fresh draws rather than the circular-shift surrogates of `02/05` because here the law with nothing to find is known and its samples carry no time order: we can draw the null itself, where a surrogate has to approximate it from a single record. The two ensembles of a null pair share their reference phase and have independent phase differences, as the two ensembles of an observed pair do. With 100 draws of each kind we compute three numbers.

- The **signed score** is the mean observed difference divided by the standard deviation of the null differences. The sign is kept on both sides, so a statistic that moved the wrong way would show a negative score.
- The **area under the curve** (AUC) is the probability that a randomly chosen high ensemble has a larger value than a randomly chosen low one (Hanley & McNeil 1982). It is 0.5 when the statistic cannot tell them apart and 1 when it always can, and it does not depend on the unit of the statistic.
- The **detection rate** is the fraction of observed differences that fall outside the **detection band**, the central 95 % of the null differences. That band is the test a practitioner would run at a false-alarm rate of 5 %.

The score says how far a statistic moves in units of its own noise. It is **not** comparable from one statistic to another: each has its own scale and responds to the moments in its own nonlinear way. The AUC and the detection rate are comparable, and they are what we read down a column.

The first cell below defines the comparison. The second computes the three numbers by hand, for the rich QMI and on a small comparison of 20,000 samples per ensemble, so that nothing afterwards is a black box.

In [ ]:
STATISTICS = {
    "PLV": plv,
    "second-harmonic PLV": lambda ta, tb: harmonic_plv(ta, tb, harmonic=2),
    "naive QMI": lambda ta, tb: coupling_qmi(joint_density_matrix(ta, tb)),
    "rich QMI": rich_qmi,
}
N_RECORDS = 100  # ensembles per set: the detection band below is calibrated on 100 null values
N_SEEDS = 5      # independent repetitions of the whole comparison
SIZES = [200, 2_000, 20_000, 50_000, N]  # samples per ensemble; section 7 uses the smaller ones
FALSE_ALARM_RATE = 0.05  # fraction of null differences allowed outside the detection band

# matched_plv_ensembles consumes seed, seed + 1 and seed + 2. Every call below gets its own
# multiple of 10, drawn without replacement, so no two calls of sections 6 and 7 share a
# random stream. (Within one call the two ensembles share their reference phase by design.)
seed_table = 10 * np.random.default_rng(15).choice(
    10**8, size=(len(SIZES), N_SEEDS, 2, N_RECORDS), replace=False
)


def run_comparison(n, seeds_observed, seeds_null):
    """Draw N_RECORDS (low, high) pairs and N_RECORDS (low, low) pairs of n samples each.

    Returns, for each statistic, its raw values on the low and high ensembles, the
    record-by-record differences high - low ('observed'), and the differences between the
    two low ensembles of a null pair ('null'), all as arrays of shape (N_RECORDS,).
    """
    values = {name: np.empty((4, N_RECORDS)) for name in STATISTICS}  # low, high, null 1, null 2
    for i in range(N_RECORDS):
        low, high = matched_plv_ensembles(
            n, a1=0.4, a2_low=0.0, a2_high=0.3, seed=int(seeds_observed[i])
        )
        null_1, null_2 = matched_plv_ensembles(
            n, a1=0.4, a2_low=0.0, a2_high=0.0, seed=int(seeds_null[i])
        )
        for name, statistic in STATISTICS.items():
            values[name][:, i] = [statistic(*pair) for pair in (low, high, null_1, null_2)]
    return {
        name: {"low": v[0], "high": v[1], "observed": v[1] - v[0], "null": v[3] - v[2]}
        for name, v in values.items()
    }

In [ ]:
# One comparison at 20,000 samples per ensemble, 100 draws of each kind (a few seconds).
# These are the seeds of the first of the five repetitions that section 7 runs at this size.
run_small = run_comparison(20_000, seed_table[2, 0, 0], seed_table[2, 0, 1])
r = run_small["rich QMI"]

# 1. Signed score: the mean observed difference, in units of the scatter of the null differences.
score_by_hand = r["observed"].mean() / r["null"].std(ddof=1)

# 2. Area under the curve: over all 100 x 100 pairs (one low ensemble, one high ensemble),
#    the fraction of pairs in which the high ensemble has the larger value.
auc_by_hand = (r["high"][:, None] > r["low"][None, :]).mean()

# 3. Detection rate: the fraction of observed differences outside the detection band, the
#    central 95 % of the null differences. method="weibull" is the quantile rule of Hyndman &
#    Fan (1996, definition 6); with 100 null values it keeps the false-alarm rate close to 5 %.
band_low, band_high = np.quantile(
    r["null"], [FALSE_ALARM_RATE / 2, 1 - FALSE_ALARM_RATE / 2], method="weibull"
)
rate_by_hand = ((r["observed"] < band_low) | (r["observed"] > band_high)).mean()

print("rich QMI, 20,000 samples per ensemble, one repetition of 100 draws")
print(f"  mean observed difference   {r['observed'].mean():8.5f} nat")
print(f"  scatter of the null        {r['null'].std(ddof=1):8.5f} nat")
print(f"  detection band             [{band_low:.5f}, {band_high:.5f}] nat")
print()
library = (
    signed_score(r["observed"], r["null"]),
    auc(r["low"], r["high"]),
    detection_rate(r["observed"], r["null"], false_alarm_rate=FALSE_ALARM_RATE),
)
print(f"{'':<22}{'by hand':>10}{'library':>10}")
for label, by_hand, from_library in zip(
    ("signed score", "area under the curve", "detection rate"),
    (score_by_hand, auc_by_hand, rate_by_hand),
    library,
):
    print(f"{label:<22}{by_hand:>10.3f}{from_library:>10.3f}")

**Read the output.** Take the three numbers one at a time. At 20,000 samples the rich QMI moves by 0.0052 nat on average between the low and the high ensemble, and its null differences scatter by 0.0023 nat: the score is their ratio, 2.3. A high ensemble has a larger rich QMI than a low one in 98 % of the 10,000 pairs: that is the area under the curve. The detection band runs from −0.0046 to +0.0047 nat, and 64 of the 100 observed differences fall outside it: a detection rate of 0.64. The library functions `signed_score`, `auc` and `detection_rate` return the same three numbers, so from here on we call them.

An almost perfect ranking (0.98) and a test that fires two times in three (0.64): the two comparable numbers do not measure the same thing, which is why we keep both.

Now the full-size comparison: all four statistics at 150,000 samples, repeated five times with fresh seeds so that every number comes with its spread. The cell takes one to two minutes and prints a line after each repetition. This is the evidence the result rests on, so it is worth the wait.

In [ ]:
def summarise(run):
    """Signed score, area under the curve and detection rate of each statistic."""
    return {
        name: (
            signed_score(r["observed"], r["null"]),
            auc(r["low"], r["high"]),
            detection_rate(r["observed"], r["null"], false_alarm_rate=FALSE_ALARM_RATE),
        )
        for name, r in run.items()
    }


def print_table(summary, title):
    """Print mean [min, max] over repetitions of the three numbers, one row per statistic."""
    print(title)
    print(f"{'statistic':<21}{'signed score':>25}{'area under the curve':>25}{'detection rate':>25}")
    for name, table in summary.items():
        cells = [f"{col.mean():7.2f} [{col.min():6.2f}, {col.max():6.2f}]" for col in table.T]
        print(f"{name:<21}" + "".join(f"{cell:>25}" for cell in cells))


# The full-size comparison: it draws 2,000 ensembles of 150,000 samples.
start = time.perf_counter()
runs_full = []
for s in range(N_SEEDS):
    runs_full.append(run_comparison(N, seed_table[-1, s, 0], seed_table[-1, s, 1]))
    print(f"repetition {s + 1} of {N_SEEDS} done ({time.perf_counter() - start:5.1f} s elapsed)")

summary_by_size = {
    N: {name: np.array([summarise(run)[name] for run in runs_full]) for name in STATISTICS}
}
print()
print_table(
    summary_by_size[N],
    f"{N:,} samples per ensemble, {N_RECORDS} ensembles per set, "
    f"mean [min, max] over {N_SEEDS} repetitions",
)

**Read the output.** The brackets give the smallest and the largest of the five repetitions.

The two first-moment statistics do not see the difference. PLV and the naive QMI have a score indistinguishable from zero (−0.07 here, a sign that belongs to the noise), an AUC of 0.48 — chance — and a detection rate of 0.05, the false-alarm rate itself. Their two rows agree almost to the digit, and that is not a bug: to sampling error the naive QMI is a function of PLV alone (`04/13`), so it orders the ensembles as PLV does.

The richer quantum measure sees it, every time: a score of 5.9 (between 5.4 and 6.3 across repetitions), an AUC of 1.00 and a detection rate of 1.00. The separation of section 4 was not sampling luck, and you have earned that sentence.

The second-harmonic PLV sees it as well: an AUC of 1.00, a detection rate of 1.00, and a score of 176. Scores are not comparable across rows, so set that last number aside. On the two comparable columns, the classical neighbour and the richer quantum measure are **tied** at 150,000 samples: both are perfect, which says only that the task is not hard enough at this size to tell them apart. Section 7 makes it harder. First, let us look at what a perfect detection looks like for each.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

rng_jit = np.random.default_rng(0)
panels = [
    ("rich QMI", "rich-QMI difference  (nats)", COLORS["quantum"]),
    ("second-harmonic PLV", "second-harmonic PLV difference", COLORS["source"]),
]
for ax, (name, ylabel, color) in zip(axes, panels):
    first = runs_full[0][name]  # first repetition: 100 null and 100 observed differences
    # The detection band: central 95 % of the null differences (Weibull quantile rule).
    band = np.quantile(
        first["null"], [FALSE_ALARM_RATE / 2, 1 - FALSE_ALARM_RATE / 2], method="weibull"
    )
    ax.axhspan(band[0], band[1], color=COLORS["grid"], alpha=0.6, zorder=1,
               label="detection band (95 % of the null)")
    ax.axhline(0.0, color=COLORS["muted"], lw=1.0, zorder=2)
    ax.scatter(
        rng_jit.uniform(-0.12, 0.12, size=N_RECORDS), first["null"], s=28,
        color=COLORS["muted"], alpha=0.8, edgecolor="white", linewidth=0.5, zorder=3,
        label="low minus low (null)",
    )
    ax.scatter(
        1.0 + rng_jit.uniform(-0.12, 0.12, size=N_RECORDS), first["observed"], s=28,
        color=color, alpha=0.85, edgecolor="white", linewidth=0.5, zorder=3,
        label="high minus low (observed)",
    )
    ax.set_xticks([0.0, 1.0])
    ax.set_xticklabels(["same $a_2$\n(null)", "different $a_2$\n(observed)"])
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylabel(ylabel)
    ax.set_title(name, pad=10)
    ax.legend(loc="upper left", fontsize=8)

fig.suptitle("Signed differences between two ensembles, 100 draws of each kind", y=1.0)
fig.tight_layout()
plt.show()

**Read the figure.** Each dot is one draw of the first repetition. In each panel the grey column on the left holds the 100 null differences (low minus low), scattered around zero, and the shaded band covers the central 95 % of them. The coloured column on the right holds the 100 observed differences (high minus low).

On the left, the richer quantum measure. Its null differences run from −0.002 to +0.0025 nat, and every observed difference lies above the band, between 0.0027 and 0.0071 nat. The two clouds do not touch — but notice how near they come: the lowest observed dot sits less than 0.001 nat above the band. On the right, the second-harmonic PLV. Its band is about ±0.003 wide and its observed differences sit near 0.30, so on this axis the null collapses to a thin line at zero.

Both statistics pass the test at this sample size. The figure shows what the table could not: the **margin**. The richer quantum measure clears its band with little room to spare, the classical neighbour with a great deal. A margin protects a result when data become scarce, which is the next question.

## 7. How many samples does each statistic need?

One hundred and fifty thousand phase samples per ensemble is generous, and an experiment does not always have that many. So we ask how each statistic behaves when samples become scarce. We rerun the protocol of section 6 at 200, 2,000, 20,000 and 50,000 samples per ensemble, five repetitions each, and add the 150,000 of section 6. The cell takes about half a minute, most of it at the largest sizes.

The detection rate of a test when there is a real difference to find is called its *power*. To read the answer we call a test reliable when its power reaches 0.8. This is the usual convention, not a quantity derived from this problem: a ruler rather than a law.

In [ ]:
for k, n in enumerate(SIZES[:-1]):
    start = time.perf_counter()
    runs = [run_comparison(n, seed_table[k, s, 0], seed_table[k, s, 1]) for s in range(N_SEEDS)]
    summary_by_size[n] = {
        name: np.array([summarise(run)[name] for run in runs]) for name in STATISTICS
    }
    print(f"{n:>7,} samples per ensemble: {N_SEEDS} repetitions done "
          f"in {time.perf_counter() - start:5.1f} s")

for column, title in ((2, "detection rate"), (1, "area under the curve")):
    print(f"\n{title}, mean [min, max] over {N_SEEDS} repetitions")
    print(f"{'samples':>9}" + "".join(f"{name:>24}" for name in STATISTICS))
    for n in SIZES:
        cells = [summary_by_size[n][name][:, column] for name in STATISTICS]
        print(f"{n:>9,}" + "".join(
            f"{f'{c.mean():.2f} [{c.min():.2f}, {c.max():.2f}]':>24}" for c in cells
        ))

**Read the output.** Read the detection-rate table one column at a time.

PLV and the naive QMI stay near the false-alarm rate, between 0.04 and 0.08, at every size. More data does not help a statistic that does not carry the information.

The second-harmonic PLV detects the difference in 99 % of the draws with **200 samples**, the smallest size we tried, and in all of them from 2,000 samples on.

The richer quantum measure needs far more. Its detection rate is 0.08 at 200 samples, at the false-alarm level, and 0.12 at 2,000, above that level but far from reliable; then 0.54 at 20,000, 0.89 at 50,000 and 1.00 at 150,000. It crosses the 0.8 line between 20,000 and 50,000 samples.

So on these two ensembles the richer quantum measure needs more than 20,000 samples to become reliable, where the classical neighbour needs 200 or fewer: **at least a hundred times more data** for the same detection. This is the most informative measurement of the notebook, and it exists only because you built a fair comparison. The Challenge lets you find out why the quantum measure pays this price here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True)

styles = {
    "PLV": (COLORS["muted"], "-"),
    "naive QMI": (COLORS["target"], "--"),
    "second-harmonic PLV": (COLORS["source"], "-"),
    "rich QMI": (COLORS["quantum"], "-"),
}
POWER_CONVENTION = 0.8  # usual convention for statistical power, not derived from this problem
panels = [(1, "area under the curve", 0.5, "chance"),
          (2, "detection rate", POWER_CONVENTION, "0.8, the usual power convention")]
for ax, (column, ylabel, reference, reference_label) in zip(axes, panels):
    ax.axhline(reference, color=COLORS["muted"], lw=1.2, ls=":", zorder=1, label=reference_label)
    for name, (color, linestyle) in styles.items():
        table = np.array([summary_by_size[n][name][:, column] for n in SIZES])  # (sizes, seeds)
        ax.fill_between(SIZES, table.min(axis=1), table.max(axis=1), color=color, alpha=0.18,
                        zorder=2)
        ax.plot(SIZES, table.mean(axis=1), color=color, ls=linestyle, lw=2.0, marker="o",
                ms=5, zorder=3, label=name)
    ax.set_xscale("log")
    ax.set_xlabel("samples per ensemble")
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.03, 1.03)
axes[0].legend(loc="lower right", fontsize=8)
axes[1].legend(loc="center left", fontsize=8)
fig.suptitle("How many samples each statistic needs to tell the two ensembles apart", y=1.0)
fig.tight_layout()
plt.show()

**Read the figure.** On the left the area under the curve, on the right the detection rate, both against the number of samples per ensemble on a logarithmic axis. Each line is the mean over five repetitions, and each shaded band runs from the smallest repetition to the largest.

The left panel adds something the table hid: the rich QMI leaves chance well before it becomes reliable. Its area under the curve is already 0.75 at 2,000 samples, where its detection rate is still 0.12. The lines of PLV and of the naive QMI lie on top of each other, the dashed one drawn over the solid one.

Three behaviours, one per kind of statistic: blind (the first moment only), sighted with a wide margin (the classical second-harmonic statistic), and sighted with a narrow margin (the richer quantum measure). This picture is the result of the notebook, and you measured every point of it.

## Your turn

**Warm-up.** Section 4 used $a_2 = 0.3$ for the high ensemble. Predict what happens to the second-harmonic PLV gap and to the rich-QMI gap if you lower it to $a_2 = 0.1$. Then check, averaging each gap over 20 draws of 150,000 samples: one draw is not enough, because the rich-QMI gap you are looking for is about the size of its own scatter. Look at its sign.

**Go further.** The null of section 6 compares two *low* ensembles ($a_2 = 0.0$ against $0.0$). Which of the three numbers of section 6 depend on the null, and which one does not? Edit the null call inside `run_comparison` so that it compares two *high* ensembles ($a_2 = 0.3$ against $0.3$), rerun at 20,000 samples, and compare the new detection band with the one of section 6.

**Challenge.** Find out why the rich QMI needs so many samples here. With $a_1 = 0.4$, average the rich-QMI gap and the second-harmonic PLV gap over 20 draws of 150,000 samples for $a_2^{\text{high}} = 0.05,\ 0.10,\ 0.17,\ 0.20$ and $0.30$, and plot both against $a_2^{\text{high}}$. One of the two gaps grows steadily; the other does not. Where is the rich-QMI gap negative, and where does it vanish? What would section 6 have reported if the high ensemble had been built with $a_2 = 0.17$?

## What you built

You ran a controlled experiment and you scored it honestly. That is the method of the capstone, and it is now yours.

- You constructed two ensembles with **identical PLV by construction** ($a_1 = 0.4$) and a **different second moment** ($a_2 = 0.0$ against $0.3$).
- You confirmed that both first-moment views are blind: **PLV** and the **naive embedding's QMI** have a score indistinguishable from zero and an AUC of 0.48 at 150,000 samples, and a detection rate at the false-alarm level at every size.
- You showed that the **richer embedding's QMI separates** them: a signed score of 5.9 (5.4 to 6.3 over five repetitions), an AUC of 1.00 and a detection rate of 1.00 at 150,000 samples.
- You set the **classical neighbour** beside it: the second-harmonic PLV separates the same ensembles, and it is reliable with 200 samples where the richer quantum measure needs more than 20,000.

State the result precisely, and no further. **On these hand-built ensembles, the richer quantum measure reads a difference that the phase-locking value cannot — and a classical second-harmonic statistic reads the same difference with at least a hundred times fewer samples.** What separates the sighted statistics from the blind ones is the harmonic they look at, not whether they are quantum: the embedding decides what a quantum measure can see, as `04/13` taught you.

Two limits remain, and they lead to the next notebook. First, we tested one value of $a_1$ and one pair of values of $a_2$, and the two ensembles differ in exactly the moment the second-harmonic PLV reads: no comparison could suit it better. The hundredfold ratio belongs to this pair, and the Challenge shows how much the picture depends on it. Second, these ensembles are drawn from a density we wrote by hand, with $a_2$ as a dial we turn ourselves. Coupled oscillators offer no such dial: their moments come out of the dynamics. In `16_capstone_synthesis` we carry the comparison to a pair of coupled oscillators whose second harmonic is free to move, and we write the accounting of the capstone from what that bench measures.

## References

- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value $\mathrm{PLV} = |\langle e^{i\Delta\theta}\rangle|$, the first circular moment that is matched across the two ensembles by construction.
- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments of a phase distribution; the first and second are distinct functionals, which lets us match PLV while parting the second moment; the modulus of the sample second moment is the statistic of section 5.
- J. A. Hanley & B. J. McNeil, "The meaning and use of the area under a receiver operating characteristic (ROC) curve", *Radiology* **143**, 29 (1982). DOI:10.1148/radiology.143.1.7063747 — the area under the receiver operating characteristic (ROC) curve, read as a probability of correct ranking.
- R. J. Hyndman & Y. Fan, "Sample quantiles in statistical packages", *The American Statistician* **50**(4), 361–365 (1996). DOI:10.1080/00031305.1996.10473566 — the quantile rule (definition 6, numpy's `weibull`) of the detection band.
- D. Trevisan, "Quantum optimal transport: an invitation", *Bollettino dell'Unione Matematica Italiana* **18**, 347–360 (2025). DOI:10.1007/s40574-024-00428-5 — the survey of quantum optimal-transport formulations behind this module.

**Previous:** `notebooks/04_QuantumOptimalTransport/14_richer_embedding_applied.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/16_capstone_synthesis.ipynb`